# TTU Data Agent Intro

### Lab Setup - Semantic Model

This notebook works with the Power BI API.  It will download and validate a semantic model from a github repositor, and updload it to your workspace.

To get started, ensure your workspace is attached to a capacity, and "Run All"


In [ ]:
import hashlib
import io
import json
import time
from urllib.parse import quote
from uuid import UUID

import notebookutils
import requests


REPOSITORY = "pawarbi/fda-l400"
REPOSITORY_REF = "main"
EXPECTED_FOLDER_NAME = "data-agent-l400"
RAW_BASE_URL = (
    f"https://raw.githubusercontent.com/{REPOSITORY}/{REPOSITORY_REF}"
)
IMPORT_TIMEOUT_MINUTES = 10
POLL_INTERVAL_SECONDS = 5

# This is 
PBIX_ASSETS = [
    {
        "file_name": "ManufacturingOps.pbix",
        "sha256": "BFC0F3EA44F51EE5A3BE0739BDF268EE298336066CC734306594B4C6B1428F09",
    }
]

for asset in PBIX_ASSETS:
    asset["path"] = f"assets/pbix/{asset['file_name']}"
    asset["url"] = f"{RAW_BASE_URL}/{quote(asset['path'], safe='/')}"

print("Source:", f"{REPOSITORY}@{REPOSITORY_REF}")
for asset in PBIX_ASSETS:
    print("PBIX:", asset["url"])

StatementMeta(, 8c207c3d-537c-4798-907d-be7831154c99, 10, Finished, Available, Finished, False)

Source: pawarbi/fda-l400@v1.0.1
PBIX: https://raw.githubusercontent.com/pawarbi/fda-l400/v1.0.1/assets/pbix/ManufacturingOps.pbix


In [9]:
def require_uuid(value, label):
    try:
        return str(UUID(str(value)))
    except (TypeError, ValueError) as exc:
        raise RuntimeError(f"Invalid or missing {label}: {value!r}") from exc


runtime_context = notebookutils.runtime.context
workspace_id = require_uuid(
    runtime_context.get("currentWorkspaceId"),
    "currentWorkspaceId in notebook runtime context",
)
notebook_id = require_uuid(
    runtime_context.get("currentNotebookId"),
    "currentNotebookId in notebook runtime context",
)

# The NotebookUtils `pbi` audience is documented for both Power BI and Fabric
# REST APIs. Acquire separate token values so each API call path is explicit.
power_bi_token = notebookutils.credentials.getToken("pbi")
fabric_api_token = notebookutils.credentials.getToken("pbi")
if not power_bi_token:
    raise RuntimeError("Power BI API token acquisition returned an empty token.")
if not fabric_api_token:
    raise RuntimeError("Fabric API token acquisition returned an empty token.")

fabric_headers = {
    "Authorization": f"Bearer {fabric_api_token}",
    "Content-Type": "application/json",
}
notebook_item_url = (
    "https://api.fabric.microsoft.com/v1/workspaces/"
    f"{workspace_id}/items/{notebook_id}"
)
notebook_item_response = requests.get(
    notebook_item_url,
    headers=fabric_headers,
    timeout=60,
)
if notebook_item_response.status_code != 200:
    raise RuntimeError(
        "Could not discover the InstallWorkshopAssets folder through the "
        f"Fabric Core item API: HTTP {notebook_item_response.status_code} "
        f"{notebook_item_response.text}"
    )

# Figure out the context of where we're running, and if we're in a sub folder of the workspace.
notebook_item = notebook_item_response.json()
folder_id_value = notebook_item.get("folderId")

folder_url = (
    "https://api.fabric.microsoft.com/v1/workspaces/"
    f"{workspace_id}/"
)

# if we're in a sub fodler, append the folder id to the target url
if folder_id_value == "None":
    folder_url = folder_url + f"folders/{folder_id}"

folder_response = requests.get(folder_url, headers=fabric_headers, timeout=60)
if folder_response.status_code != 200:
    raise RuntimeError(
        f"Could not read notebook folder: "
        f"HTTP {folder_response.status_code} {folder_response.text}"
    )
folder = folder_response.json()
folder_name = folder.get("displayName", "")

print("Target workspace:", workspace_id)
print("Current notebook:", f"{notebook_item.get('displayName', '')} ({notebook_id})")
print("Target folder:", folder_id_value)
print("Target upload url:", folder_url)

StatementMeta(, 8c207c3d-537c-4798-907d-be7831154c99, 11, Finished, Available, Finished, False)

Target workspace: a2fb2918-bd6a-492e-96a5-f25877175d79
Current notebook: ttu-setup (e4c31a52-0ffe-4dfd-970d-969b41077559)
Target folder: None
Target upload url: https://api.fabric.microsoft.com/v1/workspaces/a2fb2918-bd6a-492e-96a5-f25877175d79/


In [10]:
def download_pbix(asset):
    response = requests.get(
        asset["url"],
        headers={"User-Agent": "fabric-data-agent-l400-installer"},
        timeout=180,
    )
    response.raise_for_status()
    content = response.content
    if content.startswith(b"version https://git-lfs.github.com/spec/v1"):
        raise RuntimeError(
            f"{asset['path']} resolved to a Git LFS pointer instead of PBIX content."
        )

# Perform integrity check on the PBIX.
    actual_hash = hashlib.sha256(content).hexdigest().upper()
    if actual_hash != asset["sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {asset['file_name']}: "
            f"expected {asset['sha256']}, received {actual_hash}."
        )
    print(
        f"Downloaded and validated: {asset['file_name']} "
        f"({len(content):,} bytes)"
    )
    return content

# PBIX import API is an asych operation.  This handles initiating the import
# checking status, and executing a nubmer of retries before timing out.
def import_pbix(asset, pbix_content):
    file_name = asset["file_name"]
    import_url = (
        "https://api.powerbi.com/v1.0/myorg/groups/"
        f"{workspace_id}/imports"
        f"?datasetDisplayName={quote(file_name)}"
        "&nameConflict=CreateOrOverwrite"
    )
    response = requests.post(
        import_url,
        headers={"Authorization": f"Bearer {power_bi_token}"},
        files={
            "file": (
                file_name,
                io.BytesIO(pbix_content),
                "application/octet-stream",
            )
        },
        timeout=600,
    )
    if response.status_code not in (200, 201, 202):
        raise RuntimeError(
            f"PBIX upload failed for {file_name}: "
            f"HTTP {response.status_code} {response.text}"
        )

    try:
        import_id = response.json()["id"]
    except (KeyError, TypeError, ValueError) as exc:
        raise RuntimeError(
            f"PBIX upload for {file_name} did not return an import ID: "
            f"HTTP {response.status_code} {response.text}"
        ) from exc

    print(f"Upload accepted: {file_name} (import ID: {import_id})")
    status_url = (
        "https://api.powerbi.com/v1.0/myorg/groups/"
        f"{workspace_id}/imports/{import_id}"
    )
    attempts = max(
        1,
        IMPORT_TIMEOUT_MINUTES * 60 // POLL_INTERVAL_SECONDS,
    )
    last_state = "Unknown"
    for _ in range(attempts):
        status_response = requests.get(
            status_url,
            headers={"Authorization": f"Bearer {power_bi_token}"},
            timeout=60,
        )
        status_response.raise_for_status()
        import_result = status_response.json()
        state = import_result.get("importState", "Unknown")
        if state != last_state:
            print(f"Import state for {file_name}: {state}")
            last_state = state

        if state == "Succeeded":
            datasets = import_result.get("datasets", [])
            reports = import_result.get("reports", [])
            if not datasets or not reports:
                raise RuntimeError(
                    f"PBIX import {import_id} succeeded for {file_name}, but "
                    "the response did not include both semantic model and report "
                    f"IDs. Import details: {json.dumps(import_result)}"
                )
            artifacts = []
            for item_type, imported_items in (
                ("SemanticModel", datasets),
                ("Report", reports),
            ):
                for imported_item in imported_items:
                    item_id = imported_item.get("id")
                    if not item_id:
                        raise RuntimeError(
                            f"PBIX import {import_id} omitted an expected "
                            f"{item_type} ID for {file_name}. "
                            f"Import details: {json.dumps(import_result)}"
                        )
                    artifacts.append(
                        {
                            "id": require_uuid(
                                item_id,
                                f"{item_type} ID returned by import {import_id}",
                            ),
                            "name": imported_item.get("name", ""),
                            "type": item_type,
                            "importId": import_id,
                            "sourceFile": file_name,
                        }
                    )
            return {
                "file": file_name,
                "importId": import_id,
                "artifacts": artifacts,
                "status": state,
                "details": import_result,
            }
        if state == "Failed":
            raise RuntimeError(
                f"PBIX import failed for {file_name} (import ID: {import_id}): "
                f"{json.dumps(import_result)}"
            )
        time.sleep(POLL_INTERVAL_SECONDS)

    raise TimeoutError(
        f"PBIX import timed out for {file_name} after "
        f"{IMPORT_TIMEOUT_MINUTES} minutes (import ID: {import_id}, "
        f"last state: {last_state})."
    )


import_results = []

# Iterate the list of PBIX assets to stage to the environment, and import them.
for pbix_asset in PBIX_ASSETS:
    print(f"\nStarting PBIX import: {pbix_asset['file_name']}")
    content = download_pbix(pbix_asset)
    result = import_pbix(pbix_asset, content)
    import_results.append(result)
    imported_summary = ", ".join(
        f"{artifact['type']} {artifact['name'] or artifact['id']}"
        for artifact in result["artifacts"]
    )
    print(f"Import succeeded: {result['file']} -> {imported_summary}")

if len(import_results) != len(PBIX_ASSETS):
    raise RuntimeError(
        f"Only {len(import_results)} of {len(PBIX_ASSETS)} PBIX imports succeeded."
    )

StatementMeta(, 8c207c3d-537c-4798-907d-be7831154c99, 12, Finished, Available, Finished, False)


Starting PBIX import: ManufacturingOps.pbix
Downloaded and validated: ManufacturingOps.pbix (3,028,890 bytes)
Upload accepted: ManufacturingOps.pbix (import ID: 2080e0fc-b046-4545-849c-9dca61157d8b)
Import state for ManufacturingOps.pbix: Publishing
Import state for ManufacturingOps.pbix: Succeeded
Import succeeded: ManufacturingOps.pbix -> SemanticModel ManufacturingOps, Report ManufacturingOps
